<table width="100%"><tr style="background-color:white;">
    <td style="text-align:left;padding:0px;width:142px'">
        <a href="https://qworld.net" target="_blank">
            <img src="../images/QWorld.png"></a></td>
    <td width="*">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</td>
    <!-- ############################################# -->
    <td style="padding:0px;width:90px;">
        <img align="right" src="../images/follow_us.png" height="40px"></td>
    <td style="padding:0px;width:40px;">
        <a href="https://twitter.com/QWorld19" target="_blank">
        <img align="right" src="../images/Twitter.png" width="40px"></a> </td>
    <td style="padding:0px;width:5px;"></td>
    <td style="padding:0px;width:40px;">
        <a href="https://www.facebook.com/qworld19/" target="_blank">
        <img align="right" src="../images/Fb.png"></a></td>
    <td style="padding:0px;width:5px;"></td>
    <td style="padding:0px;width:40px;">
        <a href="https://www.linkedin.com/company/qworld19" target="_blank">
        <img align="right" src="../images/LinkedIn.png"></a></td>
    <td style="padding:0px;width:5px;"></td>
    <td style="padding:0px;width:40px;">
        <a href="https://www.youtube.com/qworld19" target="_blank">
        <img align="right" src="../images/YT.png"></a></td>
    <!-- ############################################# -->
    <td style="padding:0px;width:60px;">
        <img align="right" src="../images/join.png" height="40px"></td>
    <td style="padding:0px;width:40px;">
        <a href="https://discord.gg/akCvr7U87g"
           target="_blank">
        <img align="right" src="../images/Discord.png"></a></td>
    <!-- ############################################# -->
    <td style="padding:0px;width:72px;">
        <img align="right" src="../images/w3.png" height="40px"></td>
    <td style="padding:0px;width:40px;">
        <a href="https://qworld.net" target="_blank">
        <img align="right" src="../images/www.png"></a></td>
</tr></table>

<table width = "100%">
  <tr style="background-color:white;">
    <td style="text-align:right;vertical-align:bottom;font-size:12px;"> 
        Prepared by Anastasija Trizna (QPoland, QLatvia)</td>
    </tr>
    <tr><td align="right" style="color:#bbbbbb;background-color:#ffffff;font-size:11px;font-style:italic;">
        This cell contains some macros. If there is a problem with displaying mathematical formulas, please run this cell to load these macros.
    </td></tr>
 </table>
 
$ \newcommand{\bra}[1]{\langle #1|} $
$ \newcommand{\ket}[1]{|#1\rangle} $
$ \newcommand{\braket}[2]{\langle #1|#2\rangle} $
$ \newcommand{\dot}[2]{ #1 \cdot #2} $
$ \newcommand{\biginner}[2]{\left\langle #1,#2\right\rangle} $
$ \newcommand{\mymatrix}[2]{\left( \begin{array}{#1} #2\end{array} \right)} $
$ \newcommand{\myvector}[1]{\mymatrix{c}{#1}} $
$ \newcommand{\myrvector}[1]{\mymatrix{r}{#1}} $
$ \newcommand{\mypar}[1]{\left( #1 \right)} $
$ \newcommand{\mybigpar}[1]{ \Big( #1 \Big)} $
$ \newcommand{\sqrttwo}{\frac{1}{\sqrt{2}}} $
$ \newcommand{\dsqrttwo}{\dfrac{1}{\sqrt{2}}} $
$ \newcommand{\onehalf}{\frac{1}{2}} $
$ \newcommand{\donehalf}{\dfrac{1}{2}} $
$ \newcommand{\hadamard}{ \mymatrix{rr}{ \sqrttwo & \sqrttwo \\ \sqrttwo & -\sqrttwo }} $
$ \newcommand{\vzero}{\myvector{1\\0}} $
$ \newcommand{\vone}{\myvector{0\\1}} $
$ \newcommand{\vhadamardzero}{\myvector{ \sqrttwo \\  \sqrttwo } } $
$ \newcommand{\vhadamardone}{ \myrvector{ \sqrttwo \\ -\sqrttwo } } $
$ \newcommand{\myarray}[2]{ \begin{array}{#1}#2\end{array}} $
$ \newcommand{\X}{ \mymatrix{cc}{0 & 1 \\ 1 & 0}  } $
$ \newcommand{\Z}{ \mymatrix{rr}{1 & 0 \\ 0 & -1}  } $
$ \newcommand{\Htwo}{ \mymatrix{rrrr}{ \frac{1}{2} & \frac{1}{2} & \frac{1}{2} & \frac{1}{2} \\ \frac{1}{2} & -\frac{1}{2} & \frac{1}{2} & -\frac{1}{2} \\ \frac{1}{2} & \frac{1}{2} & -\frac{1}{2} & -\frac{1}{2} \\ \frac{1}{2} & -\frac{1}{2} & -\frac{1}{2} & \frac{1}{2} } } $
$ \newcommand{\CNOT}{ \mymatrix{cccc}{1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0} } $
$ \newcommand{\norm}[1]{ \left\lVert #1 \right\rVert } $

 ---

<h3> Noisy Channel</h3>

We will modify our <b>SendState</b> function to introduce some errors. 

`NoisyChannel` function will help us consider more real-world QKD implementation cases.

The funtion below will introduce 12.5% chance that qubit has an error.

In [9]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from random import randrange
from qiskit_aer import Aer


def NoisyChannel(qc1, qc2, qc_name):
    """
    Copy X and H gates from qc1 to qc2 and introduce noise.

    Noise model:
        - Random bit flip (X) with probability 1/7
        - Random phase flip (Z) with probability 1/7

    Parameters:
        qc1 : source QuantumCircuit
        qc2 : target QuantumCircuit
        qreg : QuantumRegister used in qc2
    """

    # --- copy gates from qc1 ---
    used_qubits = set()

    for instruction, qargs, cargs in qc1.data:
        gate_name = instruction.name

        if gate_name in ["x", "h"]:
            qubit_index = qc1.find_bit(qargs[0]).index
            used_qubits.add(qubit_index)

            if gate_name == "x":
                qc2.x(qreg[qubit_index])
            elif gate_name == "h":
                qc2.h(qreg[qubit_index])

        elif gate_name == "measure":
            # ignore measurement operations
            continue

        else:
            raise Exception(f"Unsupported instruction: {gate_name}")

    # --- introduce noise ---
    for qubit_index in used_qubits:

        # bit-flip error
        if randrange(7) < 1:
            qc2.x(qreg[qubit_index])

        # phase-flip error
        if randrange(7) < 1:
            qc2.z(qreg[qubit_index])

<div class="alert alert-block alert-warning">
    <h3>Example 1: Introducing Errors</h3>
</div>

Let's modify our code from the Notebook [Sifting and QBER Calculation](QC10_Sifting_and_QBER_calculation.ipynb) to use `NoisyChannel` and run it again to check QBER and the final key that Asja and Balvis now share.

In [10]:
def print_outcomes_in_reverse(counts): # takes a dictionary variable
    for outcome in counts: # for each key-value in dictionary
        reverse_outcome = ''
        for i in outcome: # each string can be considered as a list of characters
            reverse_outcome = i + reverse_outcome # each new symbol comes before the old symbol(s)
    return reverse_outcome

In [11]:
qreg = QuantumRegister(16) # quantum register with 16 qubits
creg = ClassicalRegister(16) # classical register with 16 bits

send=[] #Initial bit string to send
asja_basis=[] #Register to save information about encoding basis
balvis_basis=[] #Register to save information about decoding basis

#Asja
asja = QuantumCircuit(qreg, creg, name='Asja')

for i in range(16):
    bit = randrange(2)
    send.append(bit)
    
for i, n in enumerate(send):
    if n==1: asja.x(qreg[i]) # apply x-gate
        
for i in range(16):
    r=randrange(2) #Asja randomly pick a basis
    if r==0: #if bit is 0, then she encodes in Z basis
        asja_basis.append('Z')
    else: #if bit is 1, then she encodes in X basis
        asja.h(qreg[i])
        asja_basis.append('X')

balvis = QuantumCircuit(qreg, creg, name='Balvis') #Defining Balvis circuit
NoisyChannel(asja, balvis, 'Asja') #Asja sends noisy states to Balvis

#Balvis 
for i in range(16):
    r=randrange(2) #Balvis randomly pick a basis
    if r==0: #if bit is 0, then measures in Z basis
        balvis.measure(qreg[i],creg[i])
        balvis_basis.append('Z')
    else: #if bit is 1, then measures in X basis
        balvis.h(qreg[i])
        balvis.measure(qreg[i],creg[i])
        balvis_basis.append('X')

C:\Users\LAPTOP CHANNEL\AppData\Local\Temp\ipykernel_2804\975761936.py:23: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instruction, qargs, cargs in qc1.data:


In [12]:
backend = Aer.get_backend('qasm_simulator')
transpiled = transpile(balvis, backend)
job = backend.run(transpiled, shots=1)

counts = job.result().get_counts(balvis)
received = print_outcomes_in_reverse(counts)


#Sifting
asja_key=[] #Asjas register for matching rounds
balvis_key=[] #Balvis register for matching rounds
for j in range(0,len(asja_basis)): #Going through list of bases 
    if asja_basis[j] == balvis_basis[j]: #Comparing
        asja_key.append(send[j])
        balvis_key.append(received[j]) #Keeping key bit if bases matched
    else:
        pass #Discard round if bases mismatched


#QBER
rounds = len(asja_key)//3
errors=0
for i in range(rounds):
    bit_index = randrange(len(asja_key)) 
    tested_bit = asja_key[bit_index]
    if asja_key[bit_index] != balvis_key[bit_index]: #comparing tested rounds
        errors=errors+1 #calculating errors
    del asja_key[bit_index] #removing tested bits from key strings
    del balvis_key[bit_index]
QBER=errors/rounds #calculating QBER
        
print("QBER value =", QBER)
print("Asja's secret key =", asja_key)
print("Balvis' secret key =", balvis_key)

QBER value = 1.0
Asja's secret key = [1, 0, 1, 0, 0, 1]
Balvis' secret key = ['1', '0', '1', '0', '0', '1']


Re-run the previous code cell multiple times to get different QBER values since we have different randomly-generated keys.  

If you get an error: "*ValueError: invalid literal for int() with base 10*", go to "Kernel" in the menu bar at the top and select "Restart & Clear Output". That will handle the error. 

Next: [Information Reconciliation](QC14_Information_reconciliation.ipynb)